# TME 310 coding assignment 08

## Instructions

1. Solve each problem in the code cell below it. Add cells if you need them.
2. If a problem takes more than 30 minutes, stop and get help: the [course AI tutor](https://uwt-tme-310.github.io/TME_310_A/syllabus/polya_ai.html), office hours, or your group.
3. When you finish, restart the kernel, run all cells, export the notebook as HTML or PDF, and upload it on Canvas.

## Learning objectives

- Create, validate, and use models of physical systems
- Use code-based numerical tools

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt

## Problem 1

A slope shaken by an earthquake is often modeled as a rigid block resting on an incline.

Friction holds the block in place until the base acceleration toward the slope exceeds the **yield acceleration**, $a_y$. The value of the yield acceleration depends on the slope angle and the friction coefficient at the interface.

This problem and the two after it involve writing the code needed to simulate the behavior of this simplified model.

<img src="sliding_block_sine.png" width="60%">

The input is an idealized shaking of the base, a sine wave, positive toward the slope:

$$a(t) = A \sin(2\pi f t)$$

While $a \le a_y$, the block moves with the base. When $a > a_y$, friction can no longer accelerate the block as fast as the base, so the base starts to move out from under it: the block slides down the slope relative to the base, and its own acceleration stays pinned at $a_y$.

| Parameter | Value |
|---|---|
| amplitude $A$ | 5 m/s² |
| frequency $f$ | 2 Hz |
| yield acceleration $a_y$ | 2 m/s² |
| duration | 3 s |
| time step $\Delta t$ | 0.001 s |

---

1. Write `base_accel(t, A, f)` that returns $A \sin(2\pi f t)$. Make a time array from 0 to 3 s in steps of $\Delta t$ (`np.arange(0, 3, dt)`), compute the base acceleration, and plot it with a horizontal line at $a_y$.
2. Make a mask `exceeds = a > ay` and find the index at which each exceedance begins (`np.where(np.diff(exceeds.astype(int)) == 1)[0] + 1`). Print how many there are and their start times, and mark the starts on your plot. Check the first start against the exact time $t_s = \arcsin(a_y/A)\,/\,(2\pi f)$.
3. Find where the first exceedance ends the same way (the mask changes from 1 to 0) and print how long it lasts. Check it against the exact duration $\left(\pi - 2\arcsin(a_y/A)\right)/(2\pi f)$.

In [ ]:
# Your code here

## Problem 2

The block keeps sliding until the base and the block move at the same speed again, so the model has to track velocities, not only accelerations.

Instead of calculating the velocities of the block and the base separately, it is simpler to track the **relative velocity**, $v_r$, at which the base moves relative to the block (positive when the block slides down the slope).

Sliding starts at the times you found in Problem 1. From each start, the relative velocity is the integral of the difference between the base acceleration (the original signal) and the block acceleration (constant at $a_y$ during sliding). The block stops sliding when the relative velocity drops back to zero ($v_r \le 0$). It cannot slide back up the slope, so $v_r$ stays at zero until the next start. While the block is not sliding, it moves with the base:

$$a_{block} = \begin{cases} a_y & v_r > 0 \\ a & v_r = 0 \end{cases}$$

Use the sine wave and the parameters from Problem 1.

---

1. Compute the relative velocity $v_r$ at every time step.
2. Find the index at which each slide starts and stops, and print the start and stop times.
3. Plot the base and block accelerations on one set of axes and $v_r$ on a second set below it (`plt.subplots(2, 1, sharex=True)`), and mark the start and stop points on both.

In [ ]:
# Your code here

## Problem 3

The sliding displacement of the block relative to the base is the integral of the relative velocity.

A **square pulse** gives an exact answer to check the code against. The base accelerates at $A$ for a time $t_0$ and then stops accelerating. The block slides during the pulse, reaching a relative velocity $(A - a_y)\,t_0$, then keeps sliding while that velocity decreases at $a_y$, and stops at $t = t_0 A / a_y$. The total sliding displacement is

$$d = \frac{(A t_0)^2}{2 a_y}\left(1 - \frac{a_y}{A}\right)$$

| Parameter | Value |
|---|---|
| pulse amplitude $A$ | 5 m/s² |
| pulse duration $t_0$ | 0.1 s |
| yield acceleration $a_y$ | 2 m/s² |
| record length | 1 s |
| time step $\Delta t$ | 0.001 s |

---

1. Integrate the relative velocity from Problem 2 to get the sliding displacement (`cumulative_trapezoid(v, t, initial=0)`). Plot it below the relative velocity and print the final displacement in cm.
2. Make the square pulse (`np.where(t < t0, A, 0.0)`) and repeat Problem 2 and task 1 with it. Plot the acceleration, $v_r$ and the displacement on three stacked axes. Print the time the block stops and the final displacement next to the exact values.

In [ ]:
# Your code here

## Extra problem (optional)

*This problem is an [extra labor](https://uwt-tme-310.github.io/TME_310_A/extra_labor/). It is not required for the assignment to be complete. Completing it adds 0.1 to your grade (extra problems count up to 0.3 in total), and you can submit it any time before the end of the quarter. Expect it to take about two hours.*

In 2012, rock blocks resting on model slopes were shaken in a geotechnical centrifuge spinning at 20 g, which makes a small model behave like a slope 20 times larger. The data here are from Trial 10: one block on a slope of $\theta = 28.6°$, shaken with a 2 Hz sine wave (the same recording as the lesson on features from data). The values were converted to that full-size **prototype**, so times and displacements are full scale.

<img src="centrifuge_slope1.png" width="70%">

`centrifuge_trial10.csv` has a header row and four columns: time (s), the base acceleration (g), and the two block accelerometers `h` and `v` (g). Multiply the accelerations by 9.81 to work in m/s². The base accelerometer measures horizontally, signed like the earlier problems in this set (positive toward the slope). The block accelerometers are fixed to the block, so they measure in its tilted axes: `h` along the slope surface and `v` perpendicular to it. The block's **horizontal** acceleration combines the two:

$$a_x = a_h \cos\theta + a_v \sin\theta$$

The model needs $a_y$, and the record gives an estimate of it: while the block slides, its horizontal acceleration should sit at $a_y$. Take the median of the block's acceleration during the steady part of the shaking ($6 < t < 14$ s) at the samples where the base exceeds the block by more than 1 m/s².

The record also gives a **measured** sliding displacement to compare the model with: integrate the measured relative acceleration with the same stopping rule as your relative velocity code from Problem 2, with $a_y$ replaced by the measured block acceleration at each step, and integrate that velocity as in Problem 3. Use only $t \le 17.5$ s, where the shaking ends; after that, small sensor offsets would keep an unfinished slide creeping.

---

1. Load the file and plot the base acceleration and the block's `h` channel between 8 and 9.2 s.
2. **Without rotation**: treat `h` as if it were the block's horizontal acceleration. Write `estimate_ay(t, a_base, a_block)` and print the estimate. Write `measured_velocity(t, a_base, a_block)`, your relative velocity loop with `ay` replaced by `a_block[i]`. Using $t \le 17.5$ s, compute the model displacement (your Problem 2 and 3 code with the estimated $a_y$) and the measured displacement, plot both against time on one set of axes, and print the two final values in cm.
3. **With rotation**: compute $a_x$, plot it with the base and `h` over the same window as task 1, and repeat task 2 with $a_x$ as the block acceleration.

In [ ]:
# Your code here